# LightRAG

Dual-level retrieval: match specific entities (low level) and broad themes (high level), both through the graph.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

**Extra requirements:** `pip install networkx`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import networkx as nx

G = nx.Graph()
relation_text = []
for i, chunk in enumerate(chunks[:15]):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r, chunk=i)
        relation_text.append((f"{s} {r} {o}", i))

entity_store = build_store(list(G.nodes))                              # low-level index
relation_store = build_store([t for t, _ in relation_text], [{"chunk": c} for _, c in relation_text])  # high-level index
print(len(entity_store), "entities,", len(relation_store), "relations")

An LLM splits the query into **low-level keywords** (entities, details) and **high-level keywords** (themes, concepts).

In [ ]:
def keywords(query):
    out = llm_json(f'Extract keywords. Reply as JSON {{"low": [specific entities], "high": [abstract themes]}}.\n\nQuestion: {query}')
    return (out.get("low", []), out.get("high", [])) if isinstance(out, dict) else ([query], [query])

def light_rag(query, k=3):
    low, high = keywords(query)
    context = set()
    for kw in low:                                    # entity -> its relations
        for h in entity_store.search(kw, k):
            node = h["text"]
            for nb in G.neighbors(node):
                context.add(f"{node} {G[node][nb]['relation']} {nb}")
    for kw in high:                                   # theme -> matching relations -> source chunks
        for h in relation_store.search(kw, k):
            context.add(h["text"])
            context.add(chunks[h["meta"]["chunk"]])
    return low, high, answer(query, sorted(context))

low, high, response = light_rag("How do hybrid search and reranking work together to improve retrieval?")
print("low :", low); print("high:", high); print(response)